In [ ]:
import sys
sys.path.append("..")
from datetime import datetime
import torch, numpy as np

from src.environment.utils import smooth
from src.data import load_and_align_data, PAIRS, get_field

from bokeh.palettes import Category10
import bokeh.plotting as bk
bk.output_notebook()

In [ ]:
PAIRS

# Read Historical Data

In [ ]:
PAIRS_ = {
    'Bitcoin': 'XBTEUR',
    'Ethereum': 'ETHEUR',
    'Ripple': 'XRPEUR',
    'Cardano': 'ADAEUR',
    'Solana': 'SOLEUR',
}

In [ ]:
data, times = load_and_align_data(PAIRS_, interval=1)

In [ ]:
times_ = torch.tensor([t.timestamp() for t in times], dtype=torch.float64)
dt = float(times_.diff().mean().round())
print(f"dt = {dt}")
prices = torch.tensor(get_field(data, 'close')).T
volume = torch.tensor(get_field(data, 'volume')).T

history = [{
    'time'  : t,
    'prices': p,
    'volume': v
} for t, p, v in zip(times_, prices, volume)]

history = sorted(history, key=lambda x: x['time'])

len(history)

In [ ]:
# f1 = bk.figure(title=f"Prices", x_axis_type="datetime", x_axis_label="t", y_axis_label="USD", width=1200, height=500)
# max_len = 10000

# prices_sth = smooth(prices[:max_len], times_[:max_len], tau=3600)

# for i, (name, price, price_) in enumerate(zip(data.keys(), prices[:max_len].T, prices_sth.T)):
#     r = f1.line(times[:max_len], price, line_width=2, legend_label=f"{name}", color=Category10[10][i%10])
#     r = f1.line(times[:max_len], price_, line_width=2, legend_label=f"{name} smooth", color=Category10[10][i%10], alpha=0.3)
# f1.legend.click_policy = "hide"

# bk.show(f1)

In [ ]:
# f1 = bk.figure(title=f"Volumes", x_axis_type="datetime", x_axis_label="t", y_axis_label="EUR", width=1200, height=500)
# eur_vol = (volume*prices)[:max_len]
# eur_vol_sth = smooth(eur_vol, times_[:max_len], tau=60*5)

# for i, (name, vol, vol_) in enumerate(zip(data.keys(), eur_vol.T, eur_vol_sth.T)):
#     r = f1.line(times[:max_len], vol, line_width=2, legend_label=f"{name}", color=Category10[10][i%10])
#     r = f1.line(times[:max_len], vol_, line_width=2, legend_label=f"{name} smooth", color=Category10[10][i%10], alpha=0.3)
# f1.legend.click_policy = "hide"

# bk.show(f1)

In [ ]:
# f1 = bk.figure(title=f"Volumes", x_axis_type="datetime", x_axis_label="t", y_axis_label="EUR", width=1200, height=500)

# # eur_vol_rel = torch.log(eur_vol / eur_vol_sth)
# # eur_vol_rel = (eur_vol - eur_vol_sth) / eur_vol_sth
# eur_vol_rel = smooth(torch.log(eur_vol[1:] / eur_vol[:-1]), times_[1:max_len], tau=60)

# for i, (name, vol) in enumerate(zip(data.keys(), eur_vol_rel.T)):
#     r = f1.line(times[1:max_len], vol, line_width=2, legend_label=f"{name}", color=Category10[10][i%10])
# f1.legend.click_policy = "hide"

# bk.show(f1)

In [ ]:
from src.environment.proto_v06 import MultiCurrencyEnv

base_t = 60
tau_p = torch.tensor([base_t*20, base_t*60*3, base_t*60*24, base_t*60*24*7], dtype=torch.float32)
print((tau_p / (3600 * 24)).tolist(), 60*24)

env = MultiCurrencyEnv(
    N=len(data),
    C0=1_000.0,
    tau_p=tau_p,
    transaction_eps=1e-2,
    bankruptcy_threshold=10.0,
    sell_fee=1.0,
    buy_fee=1.0,
    fee_type="fixed",
    tax_rate=0.26,
    dV_coeff=1.0,
    carry_coef=0.0,
    realized_weight_cap=1.0,
)

In [ ]:
tau_p[1:] / tau_p[:-1]

In [ ]:
env.reset(history[0])

In [ ]:
env.step(None, history[910])

In [ ]:
env.state_size, env.action_size

# Integrate Agent

In [ ]:
from matplotlib.pylab import dtype
from src.network.vanilla import VanillaNetwork
from src.network.sample import VanillaQNetwork, StochasticVanillaNetwork
from src.agent.ddpg import DDPGConfig, VanillaDDPG
from src.agent.sac import SACConfig, SACAgent

agent_type = "sac"
device = "cpu"
buffer_size = 5_000_000
tau = 200.0
gamma = 0.999

if agent_type == "ddpg":
    agent = VanillaDDPG(
        actor=VanillaNetwork(env.state_size, env.action_size, sizes=[128, 128], act=torch.tanh, out_act=torch.tanh),
        critic=VanillaQNetwork(env.state_size, env.action_size, sizes=[128, 128], act=torch.tanh),
        config=DDPGConfig(
            gamma=gamma,
            tau=tau,
            noise_std=0.3,
            buffer_size=buffer_size,
            device=device
        )
    )

elif agent_type == "sac":
    from src.agent.sac import SACConfig, SACAgent, train_on_historical
    agent = SACAgent(
        actor=StochasticVanillaNetwork(env.state_size, env.action_size, sizes=[256, 256], act=torch.nn.Softsign()),
        critic=VanillaQNetwork(env.state_size, env.action_size, sizes=[256, 256], act=torch.nn.Softsign()),
        config=SACConfig(
            gamma=gamma,
            tau=tau,
            initial_alpha = 1.0,
            target_entropy = env.action_size * -0.5,
            buffer_size=buffer_size,
            device=device
        )
    )

else:
    raise NotImplementedError(f"The agent type '{agent_type}' is not implemented.")

In [ ]:
# agent.load(f"../data/agent/{agent_type}_v06.ptm")

In [ ]:
actions = {
    "no_action": torch.tensor([-10.0,   0.0,   0.0,   0.0,   0.0,   0.0,   0.0]),
    "only_cash": torch.tensor([ 10.0,  20.0, -10.0, -10.0, -10.0, -10.0, -10.0]),
    "only_BTC":  torch.tensor([ 10.0, -10.0,  20.0, -10.0, -10.0, -10.0, -10.0]),
    "only_ETH":  torch.tensor([ 10.0, -10.0, -10.0,  20.0, -10.0, -10.0, -10.0]),
    "only_XRP":  torch.tensor([ 10.0, -10.0, -10.0, -10.0,  20.0, -10.0, -10.0]),
    "only_ADA":  torch.tensor([ 10.0, -10.0, -10.0, -10.0, -10.0,  20.0, -10.0]),
    "only_SOL":  torch.tensor([ 10.0, -10.0, -10.0, -10.0, -10.0, -10.0,  20.0]),
}

In [ ]:
torch.argwhere(times_ > datetime(2023, 7, 1).timestamp())[0]

In [ ]:
agent.buffer.clear()

In [ ]:
n_steps = 60_000
offset = 0
warm_up = 10_000
state = env.reset(history[offset])

p_abss = []
p_rels = []
r_alls = []
actions_ = []

realized_costs = []
realized_pnls = []

Vs = []
for step in range(n_steps):
    _data = history[offset+step+1]
    action = actions["no_action"]

    ###### offset 0 ######

    if _data["time"].item() == datetime(2021, 6, 26, 23, 40).timestamp():
        action = actions["only_SOL"]
    if _data["time"].item() == datetime(2021, 6, 27, 5, 5).timestamp():
        action = actions["only_cash"]

    if _data["time"].item() == datetime(2021, 6, 28, 0, 0).timestamp():
        action = actions["only_SOL"]
    if _data["time"].item() == datetime(2021, 6, 28, 9, 47).timestamp():
        action = actions["only_cash"]

    if _data["time"].item() == datetime(2021, 6, 28, 16, 0).timestamp():
        action = actions["only_SOL"]
    if _data["time"].item() == datetime(2021, 6, 29, 16, 45).timestamp():
        action = actions["only_cash"]

    if _data["time"].item() == datetime(2021, 6, 30, 18, 0).timestamp():
        action = actions["only_SOL"]
    if _data["time"].item() == datetime(2021, 7, 1, 1, 40).timestamp():
        action = actions["only_cash"]

    if _data["time"].item() == datetime(2021, 7, 2, 12, 10).timestamp():
        action = actions["only_SOL"]
    if _data["time"].item() == datetime(2021, 7, 3, 12, 0).timestamp():
        action = actions["only_cash"]

    if _data["time"].item() == datetime(2021, 7, 7, 0, 0).timestamp():
        action = actions["only_SOL"]
    if _data["time"].item() == datetime(2021, 7, 7, 21, 0).timestamp():
        action = actions["only_cash"]

    if _data["time"].item() == datetime(2021, 7, 14, 11, 0).timestamp():
        action = actions["only_SOL"]
    if _data["time"].item() == datetime(2021, 7, 14, 16, 30).timestamp():
        action = actions["only_cash"]

    if _data["time"].item() == datetime(2021, 7, 21, 3, 30).timestamp():
        action = actions["only_SOL"]
    if _data["time"].item() == datetime(2021, 7, 21, 18, 39).timestamp():
        action = actions["only_cash"]

    if _data["time"].item() == datetime(2021, 7, 23, 22, 0).timestamp():
        action = actions["only_SOL"]
    if _data["time"].item() == datetime(2021, 7, 24, 8, 56).timestamp():
        action = actions["only_cash"]

    if _data["time"].item() == datetime(2021, 7, 25, 16, 45).timestamp():
        action = actions["only_SOL"]
    if _data["time"].item() == datetime(2021, 7, 26, 3, 44).timestamp():
        action = actions["only_cash"]

    ###### offset 320000 #######

    if _data["time"].item() == datetime(2022, 2, 3, 20, 45).timestamp():
        action = actions["only_XRP"]
    if _data["time"].item() == datetime(2022, 2, 5, 1, 35).timestamp():
        action = actions["only_cash"]

    if _data["time"].item() == datetime(2022, 2, 7, 2, 0).timestamp():
        action = actions["only_XRP"]
    if _data["time"].item() == datetime(2022, 2, 8, 7, 15).timestamp():
        action = actions["only_cash"]

    if _data["time"].item() == datetime(2022, 2, 8, 20, 27).timestamp():
        action = actions["only_XRP"]
    if _data["time"].item() == datetime(2022, 2, 9, 0, 15).timestamp():
        action = actions["only_cash"]

    if _data["time"].item() == datetime(2022, 2, 12, 15, 40).timestamp():
        action = actions["only_XRP"]
    if _data["time"].item() == datetime(2022, 2, 12, 18, 15).timestamp():
        action = actions["only_cash"]

    if _data["time"].item() == datetime(2022, 2, 14, 21, 0).timestamp():
        action = actions["only_XRP"]
    if _data["time"].item() == datetime(2022, 2, 15, 20, 38).timestamp():
        action = actions["only_cash"]

    if _data["time"].item() == datetime(2022, 2, 18, 20, 0).timestamp():
        action = actions["only_XRP"]
    if _data["time"].item() == datetime(2022, 2, 19, 17, 15).timestamp():
        action = actions["only_cash"]

    if _data["time"].item() == datetime(2022, 2, 22, 9, 0).timestamp():
        action = actions["only_ADA"]
    if _data["time"].item() == datetime(2022, 2, 23, 14, 15).timestamp():
        action = actions["only_cash"]

    if _data["time"].item() == datetime(2022, 2, 24, 8, 0).timestamp():
        action = actions["only_XRP"]
    if _data["time"].item() == datetime(2022, 2, 24, 21, 20).timestamp():
        action = actions["only_cash"]

    if _data["time"].item() == datetime(2022, 2, 28, 1, 40).timestamp():
        action = actions["only_ADA"]
    if _data["time"].item() == datetime(2022, 3, 1, 15, 20).timestamp():
        action = actions["only_cash"]

    # offset 1070310

    if _data["time"].item() == datetime(2023, 7, 13, 17, 10).timestamp():
        action = actions["only_XRP"]
    if _data["time"].item() == datetime(2023, 7, 13, 20, 26).timestamp():
        action = actions["only_cash"]

    if _data["time"].item() == datetime(2023, 7, 14, 4, 16).timestamp():
        action = actions["only_XRP"]
    if _data["time"].item() == datetime(2023, 7, 14, 6, 30).timestamp():
        action = actions["only_cash"]

    if _data["time"].item() == datetime(2023, 7, 14, 10, 0).timestamp():
        action = actions["only_XRP"]
    if _data["time"].item() == datetime(2023, 7, 14, 11, 0).timestamp():
        action = actions["only_cash"]

    if _data["time"].item() == datetime(2023, 7, 16, 10, 10).timestamp():
        action = actions["only_XRP"]
    if _data["time"].item() == datetime(2023, 7, 16, 12, 16).timestamp():
        action = actions["only_cash"]

    if _data["time"].item() == datetime(2023, 7, 16, 16, 00).timestamp():
        action = actions["only_XRP"]
    if _data["time"].item() == datetime(2023, 7, 16, 16, 30).timestamp():
        action = actions["only_cash"]

    if _data["time"].item() == datetime(2023, 7, 16, 23, 00).timestamp():
        action = actions["only_XRP"]
    if _data["time"].item() == datetime(2023, 7, 16, 23, 30).timestamp():
        action = actions["only_cash"]

    if _data["time"].item() == datetime(2023, 7, 19, 3, 49).timestamp():
        action = actions["only_XRP"]
    if _data["time"].item() == datetime(2023, 7, 19, 5, 13).timestamp():
        action = actions["only_cash"]

    if _data["time"].item() == datetime(2023, 7, 20, 22, 35).timestamp():
        action = actions["only_XRP"]
    if _data["time"].item() == datetime(2023, 7, 20, 23, 52).timestamp():
        action = actions["only_cash"]

    next_state, reward, done, info_ = env.step(action, _data)
    
    if step >= warm_up:
        p_abss.append(_data['prices'])
        p_rels.append(state.p_rel)
        r_alls.append(reward)
        actions_.append(info_['a'])

        realized_costs.append(env.realized_cost.clone())
        realized_pnls.append(env.realized_pnl.clone())

        Vs.append(env.V.clone())

        # agent.buffer.store(
        #     state.to_tensor().detach(),
        #     action.detach(),
        #     next_state.to_tensor().detach(),
        #     torch.tensor(reward).to(env.dtype),
        #     torch.tensor(done).to(env.dtype)
        # )

    state = next_state
    
p_abss = torch.stack(p_abss)
p_rels = torch.stack(p_rels)
r_alls = torch.tensor(r_alls)
actions_ = torch.tensor(actions_)

realized_costs = torch.stack(realized_costs)
realized_pnls = torch.stack(realized_pnls)

Vs = torch.stack(Vs)

In [ ]:
fig = bk.figure(title=f"Prices", x_axis_type="datetime", x_axis_label="t", y_axis_label="p_rel", width=1200, height=500)
for i in range(env.N):
    fig.line([datetime.fromtimestamp(history[j+1]['time'].item()) for j in range(offset<+warm_up, offset+n_steps)], p_abss[:,i] / p_abss[:,i].max(),
             line_width=2, color=Category10[10][i], legend_label=list(PAIRS_.keys())[i])
fig.legend.click_policy = "hide"
bk.show(fig)

In [ ]:
fig = bk.figure(title=f"Realized Costs", x_axis_type="datetime", x_axis_label="t", y_axis_label="€", width=1200, height=500)
for i in range(env.N):
    fig.line([datetime.fromtimestamp(history[j+1]['time'].item()) for j in range(offset+warm_up, offset+n_steps)], realized_costs[:,i],
             line_width=2, color=Category10[10][i], legend_label=f"{list(PAIRS_.keys())[i]} Realized Cost")
fig.legend.click_policy = "hide"
bk.show(fig)

In [ ]:
fig = bk.figure(title=f"Realized PnL", x_axis_type="datetime", x_axis_label="t", y_axis_label="€", width=1200, height=500)
for i in range(env.N):
    fig.line([datetime.fromtimestamp(history[j+1]['time'].item()) for j in range(offset+warm_up, offset+n_steps)], realized_pnls[:,i],
             line_width=2, color=Category10[10][i], legend_label=f"{list(PAIRS_.keys())[i]} PnL", line_dash="dashed")
fig.legend.click_policy = "hide"
bk.show(fig)

In [ ]:
fig = bk.figure(title=f"Realized RoI", x_axis_type="datetime", x_axis_label="t", y_axis_label="Realized ROI [1]", width=1200, height=500)
for i in range(env.N):
    fig.line([datetime.fromtimestamp(history[j+1]['time'].item()) for j in range(offset+warm_up, offset+n_steps)],
             torch.where(realized_costs[:,i] > 0.0, realized_pnls[:,i] / realized_costs[:,i], torch.zeros_like(realized_pnls[:,i])),
             line_width=2, color=Category10[10][i], legend_label=f"{list(PAIRS_.keys())[i]} PnL/Cost")
fig.legend.click_policy = "hide"
bk.show(fig)

In [ ]:
ts__ = [datetime.fromtimestamp(history[j+1]['time'].item()) for j in range(offset+warm_up, offset+n_steps)]

fig = bk.figure(title=f"Rewards", x_axis_type="datetime", x_axis_label="t", y_axis_label="Reward/Return", width=1200, height=500)

fig.line(ts__, r_alls, line_width=2, color=Category10[10][0], legend_label="Reward")
returns = []
gamma = 0.999
G = 0.0
for r in reversed(r_alls):
    G = r + gamma * G
    returns.insert(0, G)
fig.line(ts__, returns, line_width=2, color=Category10[10][5], legend_label="Return")
fig.legend.click_policy = "hide"
bk.show(fig)

In [ ]:
fig = bk.figure(title=f"V", x_axis_type="datetime", x_axis_label="t", y_axis_label="Portfolio Value [€]", width=1200, height=500)
fig.line([datetime.fromtimestamp(history[j+1]['time'].item()) for j in range(offset+warm_up, offset+n_steps)], Vs,
            line_width=2, color=Category10[10][0], legend_label=f"Portfolio Value")
fig.legend.click_policy = "hide"
bk.show(fig)

In [ ]:
fig = bk.figure(title=f"Actions", x_axis_type="datetime", x_axis_label="t", y_axis_label="action", width=1200, height=500)
for i in range(env.N):
    fig.line([datetime.fromtimestamp(history[j+1]['time'].item()) for j in range(offset+warm_up, offset+n_steps)], actions_[:,i],
             line_width=2, color=Category10[10][i], legend_label=list(PAIRS_.keys())[i])
fig.legend.click_policy = "hide"
bk.show(fig)

In [ ]:
loss, rewards, info = [], [], []

In [ ]:
_loss, _rewards, _info = train_on_historical(
    agent, env, history[:int(len(history)*0.9)],
    n_episodes=10, batch_size=128,
    update_interval=100, n_updates=10,
    max_steps=60000, warm_up=15000,
    actor_lr=1e-3, critic_lr=1e-3,
    optim="AdamW"
)
loss    += _loss
rewards += _rewards
info    += _info

In [ ]:
x = torch.randn([12, 10])
x.shape

In [ ]:
y = torch.randn([1, 10]).expand_as(x)
x.shape

In [ ]:
agent.save(f"../data/agent/{agent_type}_v06.ptm")

In [ ]:
# ls_a = []
# ls_c = []

In [ ]:
n_updates = 1000
for _ in range(n_updates):
    loss_dict = agent.update(batch_size=128)
    ls_a.append(loss_dict["actor_loss"])
    ls_c.append(loss_dict["critic_loss"])

fig = bk.figure(title="Actor Losses", x_axis_label="Training Iteration [Batch]", y_axis_label="Loss [1]", width=900, height=320)
fig.line(list(range(len(ls_a))), ls_a, line_width=2, color=Category10[10][0])
bk.show(fig)

fig = bk.figure(title="Critic Losses", x_axis_label="Training Iteration [Batch]", y_axis_label="Loss [1]", width=900, height=320)
fig.line(list(range(len(ls_c))), ls_c, line_width=2, color=Category10[10][2])
bk.show(fig)

In [ ]:
# loss_a = torch.tensor([[loss_dict['actor_loss'] for loss_dict in episode_loss] for episode_loss in loss])
loss_a_all = np.concatenate([np.array([loss_dict['actor_loss'] for loss_dict in episode_loss]) for episode_loss in loss])
loss_a_mean = np.array([np.mean([loss_dict['actor_loss'] for loss_dict in episode_loss]) for episode_loss in loss])
loss_a_min = np.array([np.min([loss_dict['actor_loss'] for loss_dict in episode_loss]) for episode_loss in loss])
loss_a_max = np.array([np.max([loss_dict['actor_loss'] for loss_dict in episode_loss]) for episode_loss in loss])

fig = bk.figure(title="Actor Losses", x_axis_label="Training Iteration [Epochs]", y_axis_label="Loss", width=900, height=320)
# fig.line(torch.arange(loss_a.numel()) / loss_a.shape[1], loss_a.flatten(), line_width=2, legend_label="Loss / Batch", color=Category10[10][0], alpha=0.3)
# fig.line(torch.arange(len(loss_a)) + 0.5, loss_a.mean(1), line_width=2, legend_label="Epoch Average", color=Category10[10][0])
fig.line(np.linspace(0, len(loss_a_mean), len(loss_a_all)), loss_a_all, line_width=2, legend_label="Batch Loss", color=Category10[10][0], alpha=0.3)
# fig.line(np.arange(len(loss_a_mean))+0.5, loss_a_mean, line_width=2, legend_label="Epoch Average", color=Category10[10][0])
# fig.varea(np.arange(len(loss_a_mean))+0.5, y1=loss_a_min, y2=loss_a_max, color=Category10[10][0], alpha=0.3)
bk.show(fig)

In [ ]:
# loss_c = torch.tensor([[loss_dict['critic_loss'] for loss_dict in episode_loss] for episode_loss in loss])
loss_c_all = np.concatenate([np.array([loss_dict['critic_loss'] for loss_dict in episode_loss]) for episode_loss in loss])
loss_c_mean = np.array([np.mean([loss_dict['critic_loss'] for loss_dict in episode_loss]) for episode_loss in loss])
loss_c_min = np.array([np.min([loss_dict['critic_loss'] for loss_dict in episode_loss]) for episode_loss in loss])
loss_c_max = np.array([np.max([loss_dict['critic_loss'] for loss_dict in episode_loss]) for episode_loss in loss])

fig = bk.figure(title="Critic Losses", x_axis_label="Training Iteration [Epochs]", y_axis_label="Loss [1]", width=900, height=320)
# fig.line(torch.arange(loss_c.numel()) / loss_c.shape[1], loss_c.flatten(), line_width=2, legend_label="Loss / Batch", color=Category10[10][2], alpha=0.3)
# fig.line(torch.arange(len(loss_c)) + 0.5, loss_c.mean(1), line_width=2, legend_label="Epoch Average", color=Category10[10][2])
fig.line(np.linspace(0, len(loss_c_mean), len(loss_c_all)), loss_c_all, line_width=2, legend_label="Batch Loss", color=Category10[10][2], alpha=0.3)
# fig.line(np.arange(len(loss_c_mean))+0.5, loss_c_mean, line_width=2, legend_label="Epoch Average", color=Category10[10][2])
# fig.varea(np.arange(len(loss_c_mean))+0.5, y1=loss_c_min, y2=loss_c_max, color=Category10[10][2], alpha=0.3)

bk.show(fig)

In [ ]:
# rewards = torch.tensor(rewards)
rewards_ = [sum(r) for r in rewards]

fig = bk.figure(title="Total Rewards", x_axis_label="Training Iteration [Episodes]", y_axis_label="Loss", width=900, height=320)
# fig.line(torch.arange(rewards.numel()) / rewards.shape[1], rewards.flatten(), line_width=2, legend_label="Reward", color=Category10[10][4], alpha=0.3)
# fig.line(torch.arange(len(rewards)) + 0.5, rewards.mean(1), line_width=2, legend_label="Average Reward / Episode", color=Category10[10][4])
fig.line(list(range(len(rewards_))), rewards_, line_width=2, legend_label="Total Reward / Episode", color=Category10[10][4])
bk.show(fig)

In [ ]:
# rewards = torch.tensor(rewards)
rewards_ = [sum(r) / len(r) for r in rewards]
# rewards_min = [min(r) for r in rewards]
# rewards_max = [max(r) for r in rewards]

fig = bk.figure(title="Average Rewards", x_axis_label="Training Iteration [Episodes]", y_axis_label="Loss", width=900, height=320)
# fig.line(torch.arange(rewards.numel()) / rewards.shape[1], rewards.flatten(), line_width=2, legend_label="Reward", color=Category10[10][4], alpha=0.3)
# fig.line(torch.arange(len(rewards)) + 0.5, rewards.mean(1), line_width=2, legend_label="Average Reward / Episode", color=Category10[10][4])
fig.line(list(range(len(rewards_))), rewards_, line_width=2, legend_label="Average Reward / Episode", color=Category10[10][4])
# fig.varea(list(range(len(rewards_))), y1=rewards_min, y2=rewards_max, color=Category10[10][4], alpha=0.3)
bk.show(fig)

In [ ]:
episode = -1

ts = [datetime.fromtimestamp(item['t']) for item in info[episode]]
ps = torch.tensor([item['p'] for item in info[episode]])
Vs = torch.tensor([item['V'] for item in info[episode]])
Cs = torch.tensor([item['C'] for item in info[episode]])
vs = torch.tensor([[w * p  / item['V'] for w, p in zip(item['w'], item['p'])] for item in info[episode]])

f0 = bk.figure(title=f"Episode {episode} - Prices", x_axis_label="t", y_axis_label=r"\(p / p_{max} [1]\)", x_axis_type="datetime", width=900, height=320)
for i, name in enumerate(data):
    f0.line(ts, ps[:,i] / ps[:,i].max(), line_width=2, legend_label=f"{name}", color=Category10[10][i%10])
    # f0.line(ts, smooth(ps[:,i] / ps[:,i].max(), [item['t'] for item in info[episode]], tau=60*3), line_width=2, legend_label=f"{name}", color=Category10[10][i%10])
f0.legend.click_policy = "hide"
bk.show(f0)

f0 = bk.figure(title=f"Episode {episode} - Portfolio Fraction", x_axis_label="t", y_axis_label="EUR", x_axis_type="datetime", width=900, height=320)
f0.line(ts, Cs / Vs, line_width=2, line_dash="dashed", legend_label="Cash", color=Category10[10][0])
for i, name in enumerate(data):
    f0.line(ts, vs[:,i], line_width=2, legend_label=f"{name}", color=Category10[10][i%10])
f0.legend.click_policy = "hide"
bk.show(f0)

f1 = bk.figure(title=f"Episode {episode} - Portfolio Value", x_axis_label="t", y_axis_label="EUR", x_axis_type="datetime", width=900, height=320)
r1 = f1.line(ts, Vs, line_width=2, legend_label="Total V")
r2 = f1.line(ts, Cs, line_width=1, line_dash="dashed", legend_label="Cash C")
f1.legend.click_policy = "hide"
bk.show(f1)

fig = bk.figure(title=f"Episode {episode} - Rewards", x_axis_label="t", y_axis_label=r"Reward \(\left(\log(\frac{V_{t+1}}{V_t})\right)\)", x_axis_type="datetime", width=900, height=320)
fig.scatter(ts, rewards[episode], size=2, color=Category10[10][4], legend_label="Reward")

returns = []
gamma = 0.999
G = 0.0
for r in reversed(rewards[episode]):
    G = r + gamma * G
    returns.insert(0, G)
fig.line(ts, returns, line_width=2, color=Category10[10][5], legend_label="Return")

hist, edges = torch.histogram(torch.tensor(rewards[episode]), bins=200, density=True)
x = (edges[:-1] + edges[1:]) / 2
figh = bk.figure(title="Reward Distribution", width=300, height=320)
figh.harea(y=x, x1=0, x2=hist, fill_color=Category10[10][4], fill_alpha=0.4)
figh.line(hist, x, line_color=Category10[10][4], line_width=2)
fig.legend.click_policy = "hide"

bk.show(bk.row(fig, figh))

In [ ]:
_actions = torch.tensor([item['a'] for item in info[episode]])
ts = [j for j in range(len(_actions))]

f1 = bk.figure(title=f"Action Fraction", x_axis_label="t", y_axis_label="Buy/Sell [USD]", width=900, height=320)
for i in range(env.N):
    f1.scatter(ts, _actions[:,i], size=3, legend_label=f"a_{i+1}", color=Category10[10][(i)%10])
    f1.line(ts, _actions[:,i], line_width=1, line_dash="dashed", legend_label=f"a_{i+1}", color=Category10[10][(i)%10])
f1.legend.click_policy = "hide"
bk.show(f1)

In [ ]:
raise

# Validate

In [ ]:
start = int(len(history)*0.9)

env.save_history = True

hist_s = []
hist_r = []
hist_i = []

state = env.reset(history[start])
for elem in history[start:]:
    a = agent.act(state.to_tensor(), explore=False)
    state, reward, done, _info = env.step(a, data=elem)

    hist_s.append(state)
    hist_r.append(reward)
    hist_i.append(_info)

    if done:
        break

In [ ]:
Vs = [item['V'] for item in hist_i]
Cs = [item['C'] for item in hist_i]
ts = [datetime.fromtimestamp(item["t"]) for item in hist_i]

skip = 10
f1 = bk.figure(
    title=f"Prices",
    width=1200, height=400,
    x_range=(ts[0], ts[-1]),
    x_axis_type="datetime",
    x_axis_label="t",
    y_axis_label="EUR",
)
for i, (name, price) in enumerate(zip(data.keys(), prices.T)):
    r = f1.line(times[start::skip], price[start::skip], line_width=2, legend_label=f"{name}", color=Category10[10][i%10])
f1.legend.click_policy = "hide"
bk.show(f1)

fig1 = bk.figure(
    title=f"Portfolio Value",
    width=1200, height=400,
    x_range=(ts[0], ts[-1]),
    x_axis_type="datetime",
    x_axis_label="t",
    y_axis_label="EUR",
)
fig1.line(ts, Vs, line_width=2, legend_label="Total V")
fig1.line(ts, Cs, line_width=1, line_dash="dashed", legend_label="Cash C")
fig1.legend.click_policy = "hide"
bk.show(fig1)

fig2 = bk.figure(
    title=f"Rewards",
    width=1200, height=400,
    x_range=(ts[0], ts[-1]),
    x_axis_type="datetime",
    x_axis_label="t",
    y_axis_label="Reward (log(V_t+1/V_t))",
)
fig2.line(ts, hist_r, line_width=2, color=Category10[10][4])

hist, edges = torch.histogram(torch.tensor(hist_r), bins=200, density=True)
x = (edges[:-1] + edges[1:]) / 2

figh = bk.figure(title="Reward Distribution", width=300, height=400)
figh.harea(y=x, x1=0, x2=hist, fill_color=Category10[10][4], fill_alpha=0.4)
figh.line(hist, x, line_color=Category10[10][4], line_width=2)

bk.show(bk.row(fig2, figh))

In [ ]:
for i, name in enumerate(data):
    print(i, name)

In [ ]:
skip = 100
Vs = torch.tensor([item["V"] for item in hist_i[::skip]])
Cs = torch.tensor([item["C"] for item in hist_i[::skip]])
ws = torch.stack([item["w"] for item in hist_i[::skip]])
ps = torch.stack([item["p"] for item in hist_i[::skip]])
vs = (ps * ws / Vs[:,None])

print(Vs.shape)
print(Cs.shape)
print(ws.shape)
print(ps.shape)
print(vs.shape)

f = bk.figure(
    title=f"Portfolio Fration",
    width=1200, height=400,
    x_range=(ts[0], ts[-1]),
    x_axis_type="datetime",
    x_axis_label=r"\(\text{t}\)",
    y_axis_label=r"\(\text{Fraction} [1]\)",
)
f.line(ts[::skip], Cs / Vs, line_width=2, line_dash="dashed", legend_label="Cash", color=Category10[10][0])
for i, name in enumerate(data):
    f.line(ts[::skip], vs[:,i], line_width=2, legend_label=f"{name}", color=Category10[10][i%10])
f.legend.click_policy = "hide"
bk.show(f)

skip = 10
f1 = bk.figure(
    title=f"Prices",
    width=1200, height=400,
    x_range=(ts[0], ts[-1]),
    x_axis_type="datetime",
    x_axis_label=r"\(\text{t}\)",
    y_axis_label=r"\(p / p_{max} [1]\)",
)
for i, (name, price) in enumerate(zip(data.keys(), prices.T)):
    r = f1.line(times[start::skip], price[start::skip] / price.max(), line_width=2, legend_label=f"{name}", color=Category10[10][i%10])
f1.legend.click_policy = "hide"
bk.show(f1)